# RAG Pipeline using Groq + LangChain

This notebook builds a **Retrieval-Augmented Generation (RAG)** pipeline that:
1. Fetches a YouTube video transcript
2. Splits it into chunks
3. Embeds chunks using HuggingFace (free, local)
4. Stores embeddings in a FAISS vector store
5. Answers questions using **Groq's llama-3.3-70b** model

---
**Get your free Groq API key at:** https://console.groq.com/keys

## 🔑 Set your Groq API Key

In [1]:
import os
from google.colab import userdata

# Best practice: store your key in Colab Secrets (the 🔑 icon in the left sidebar)
# and access it like this:
try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("✅ Groq API key loaded from Colab Secrets")
except Exception:
    # Fallback: paste your key directly (not recommended for shared notebooks)
    os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"
    print("⚠️  Using hardcoded key — consider using Colab Secrets instead")

✅ Groq API key loaded from Colab Secrets


## 📦 Install Libraries

In [4]:
!pip install -q \
    youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    tiktoken

print("✅ All libraries installed")

✅ All libraries installed


## 📥 Imports

In [7]:
!pip install -q \
    youtube-transcript-api \
    langchain \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    langchain-text-splitters \
    faiss-cpu \
    sentence-transformers \
    tiktoken

In [8]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

## Step 1 — Fetch YouTube Transcript

Replace `video_id` with any YouTube video ID you want to query.

*(Default: Lex Fridman's interview with Demis Hassabis of DeepMind)*

In [11]:
video_id = "HAnw168huqA"  # Replace with your video ID

try:
    # Fix for newer youtube-transcript-api versions (>=0.7.0)
    ytt_api = YouTubeTranscriptApi()
    fetched = ytt_api.fetch(video_id)
    transcript = " ".join(snippet.text for snippet in fetched)
    print(f"✅ Transcript fetched — {len(transcript)} characters")
    print("\n--- Preview (first 500 chars) ---")
    print(transcript[:500])

except TranscriptsDisabled:
    print("❌ No captions available for this video.")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Transcript fetched — 52760 characters

--- Preview (first 500 chars) ---
[MUSIC] Welcome. I'm very excited today to talk about
effective speaking in spontaneous situations. I thank you all for joining us, even though the title of my
talk is grammatically incorrect. I thought that might scare a few of you
away. But I learned teaching here at the
business school, catching people's attention is hard. So, something as simple as that, I
thought, might draw a few of you here, so this is going to be a highly interactive
and participative workshop today. If you don't feel co


## Step 2 — Split into Chunks

In [12]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.create_documents([transcript])

print(f"✅ Split into {len(chunks)} chunks")
print("\n--- Sample chunk ---")
print(chunks[0].page_content)

✅ Split into 66 chunks

--- Sample chunk ---
[MUSIC] Welcome. I'm very excited today to talk about
effective speaking in spontaneous situations. I thank you all for joining us, even though the title of my
talk is grammatically incorrect. I thought that might scare a few of you
away. But I learned teaching here at the
business school, catching people's attention is hard. So, something as simple as that, I
thought, might draw a few of you here, so this is going to be a highly interactive
and participative workshop today. If you don't feel comfortable
participating that's completely fine, but do know I'm gonna ask you to talk to
people next to you. They'll be opportunities to stand up and practice some things because I believe the
way we become effective communicators is by actually communicating, so let's get
started right away. I'd like to ask you all to read this
sentence, and as you read this sentence, what's most important to me is that you


## Step 3 — Create Embeddings & FAISS Vector Store

We use **HuggingFace's `all-MiniLM-L6-v2`** — a lightweight, high-quality embedding model that runs locally for free (no API key needed).

In [13]:
print("⏳ Loading embedding model (downloads ~90MB on first run)...")

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("⏳ Building FAISS vector store...")
vector_store = FAISS.from_documents(chunks, embedding_model)

print("✅ Vector store ready!")

⏳ Loading embedding model (downloads ~90MB on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

⏳ Building FAISS vector store...
✅ Vector store ready!


## Step 4 — Set Up Retriever

In [14]:
# Retrieve the top 4 most relevant chunks for each query
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("✅ Retriever ready")

# Quick test
test_docs = retriever.invoke("What is DeepMind working on?")
print(f"\n📄 Retrieved {len(test_docs)} chunks for test query")

✅ Retriever ready

📄 Retrieved 4 chunks for test query


## Step 5 — Set Up Groq LLM

Using **llama-3.3-70b-versatile** — Groq's highest quality model, runs at blazing speed.

In [15]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,           # 0 = deterministic, factual answers
    max_tokens=1024,
    api_key=os.environ["GROQ_API_KEY"]
)

print("✅ Groq LLM ready (llama-3.3-70b-versatile)")

✅ Groq LLM ready (llama-3.3-70b-versatile)


## Step 6 — Create Prompt Template

In [16]:
prompt_template = """
You are a helpful assistant. Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

print("✅ Prompt template ready")

✅ Prompt template ready


## Step 7 — Build the RAG Chain

Wiring everything together with LangChain Expression Language (LCEL):

```
question → [retriever | format_docs]  ──┐
           [RunnablePassthrough]    ──┘ → prompt → llm → parser → answer
```

In [17]:
def format_docs(retrieved_docs):
    """Concatenate retrieved chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in retrieved_docs)


# Step 1: Retrieve context + pass question through in parallel
parallel_chain = RunnableParallel({
    "context": retriever | RunnableLambda(format_docs),
    "question": RunnablePassthrough()
})

# Step 2: Full chain — parallel → prompt → LLM → parse output
parser = StrOutputParser()
rag_chain = parallel_chain | prompt | llm | parser

print("✅ RAG chain assembled")

✅ RAG chain assembled


## 🚀 Ask Questions!

Run the cell below with any question about the video.

In [18]:
def ask(question):
    """Ask a question about the video and print the answer."""
    print(f"❓ Question: {question}")
    print("-" * 60)
    answer = rag_chain.invoke(question)
    print(f"🤖 Answer:\n{answer}")
    print("=" * 60)
    return answer


# --- Try some questions ---
ask("summary of the video")

❓ Question: summary of the video
------------------------------------------------------------
🤖 Answer:
The video appears to be a workshop on effective speaking in spontaneous situations. The speaker discusses the importance of setting expectations, structure, and listening in communication. They introduce two structures for responding to situations: "problem, solution, benefit" (or "opportunity, solution, benefit") and "what, so what, now what". The speaker also emphasizes the need to listen and respond in a structured way, telling a story with a clear structure. The workshop is interactive, with participants engaging in exercises to practice their communication skills.


'The video appears to be a workshop on effective speaking in spontaneous situations. The speaker discusses the importance of setting expectations, structure, and listening in communication. They introduce two structures for responding to situations: "problem, solution, benefit" (or "opportunity, solution, benefit") and "what, so what, now what". The speaker also emphasizes the need to listen and respond in a structured way, telling a story with a clear structure. The workshop is interactive, with participants engaging in exercises to practice their communication skills.'

In [20]:
ask("how to talk spontaneous?")

❓ Question: how to talk spontaneous?
------------------------------------------------------------
🤖 Answer:
To talk spontaneously, follow these four steps: 
1. Get out of your own way: stop trying to be perfect and give the right answer.
2. Reframe the situation as an opportunity: see interactions as chances to give, not challenges.
3. Listen: focus on the moment and truly understand what the other person is saying before responding.
4. Tell a story with structure: respond in a way that has a clear beginning, middle, and end, using a framework such as "what, so what, now what".


'To talk spontaneously, follow these four steps: \n1. Get out of your own way: stop trying to be perfect and give the right answer.\n2. Reframe the situation as an opportunity: see interactions as chances to give, not challenges.\n3. Listen: focus on the moment and truly understand what the other person is saying before responding.\n4. Tell a story with structure: respond in a way that has a clear beginning, middle, and end, using a framework such as "what, so what, now what".'

In [21]:
ask("Can you summarize the main topics of this video?")

❓ Question: Can you summarize the main topics of this video?
------------------------------------------------------------
🤖 Answer:
The main topics of this video are: 
1. Effective speaking in spontaneous situations
2. The importance of structure in communication, such as the "problem, solution, benefit" and "what, so what, now what" structures
3. Using humor in speaking and its risks and rewards.


'The main topics of this video are: \n1. Effective speaking in spontaneous situations\n2. The importance of structure in communication, such as the "problem, solution, benefit" and "what, so what, now what" structures\n3. Using humor in speaking and its risks and rewards.'

---
## 🗺️ Pipeline Summary

| Step | Component | Tool Used |
|------|-----------|----------|
| Transcript Fetch | YouTube API | `youtube-transcript-api` |
| Chunking | Text Splitter | `RecursiveCharacterTextSplitter` |
| Embeddings | Local model | `HuggingFace all-MiniLM-L6-v2` |
| Vector Store | Similarity search | `FAISS` |
| LLM | Answer generation | `Groq llama-3.3-70b-versatile` |
| Chain | Orchestration | `LangChain LCEL` |